In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS training_catalog.gold;
USE training_catalog.gold;
     

SHOW SCHEMAS IN training_catalog;
     

USE training_catalog.gold;
     

CREATE TABLE IF NOT EXISTS DimCustomer (
    CustomerSK BIGINT GENERATED ALWAYS AS IDENTITY,
    CustomerID INT,
    CustomerName STRING,
    Email STRING,
    City STRING,
    Address STRING,
    StartDate DATE,
    EndDate DATE,
    IsActive INT
) USING DELTA;
     

CREATE TABLE IF NOT EXISTS DimProduct (
    ProductSK BIGINT GENERATED ALWAYS AS IDENTITY,
    ProductID INT,
    ProductName STRING,
    Category STRING,
    UnitPrice DECIMAL(10,2),
    EffectiveDate DATE
) USING DELTA;
     

CREATE TABLE IF NOT EXISTS DimStore (
    StoreSK BIGINT GENERATED ALWAYS AS IDENTITY,
    StoreID INT,
    StoreName STRING,
    Region STRING
) USING DELTA;
     

CREATE TABLE IF NOT EXISTS FactSales (
    SalesSK BIGINT GENERATED ALWAYS AS IDENTITY,
    TransactionID INT,
    CustomerSK BIGINT,
    ProductSK BIGINT,
    StoreSK BIGINT,
    Quantity INT,
    Amount DECIMAL(10,2),
    TxnDate DATE
) USING DELTA;
     

MERGE INTO DimCustomer t
USING training_catalog.silver.silver_customers s
ON t.CustomerID = s.CustomerID
AND t.IsActive = 1
WHEN MATCHED
AND (
       t.City <> s.City
    OR t.Address <> s.Address
)
THEN UPDATE SET
    EndDate = CURRENT_DATE(),
    IsActive = 0;
     

INSERT INTO DimCustomer
(
    CustomerID,
    CustomerName,
    Email,
    City,
    Address,
    StartDate,
    EndDate,
    IsActive
)
SELECT
    s.CustomerID,
    s.CustomerName,
    s.Email,
    s.City,
    s.Address,
    CURRENT_DATE(),
    DATE '9999-12-31',
    1
FROM training_catalog.silver.silver_customers s
LEFT JOIN DimCustomer d
ON s.CustomerID = d.CustomerID
AND d.IsActive = 1
WHERE d.CustomerID IS NULL
   OR d.City <> s.City
   OR d.Address <> s.Address;
     

SELECT COUNT(*) FROM DimCustomer;
     

MERGE INTO DimProduct t
USING training_catalog.silver.silver_products s
ON t.ProductID = s.ProductID
WHEN MATCHED THEN UPDATE SET
    t.ProductName = s.ProductName,
    t.Category = s.Category,
    t.UnitPrice = s.UnitPrice
WHEN NOT MATCHED THEN INSERT
(
    ProductID,
    ProductName,
    Category,
    UnitPrice,
    EffectiveDate
)
VALUES
(
    s.ProductID,
    s.ProductName,
    s.Category,
    s.UnitPrice,
    CURRENT_DATE()
);
     

MERGE INTO DimStore t
USING training_catalog.silver.silver_stores s
ON t.StoreID = s.StoreID
WHEN MATCHED THEN UPDATE SET
    t.StoreName = s.StoreName,
    t.Region = s.Region
WHEN NOT MATCHED THEN INSERT
(
    StoreID,
    StoreName,
    Region
)
VALUES
(
    s.StoreID,
    s.StoreName,
    s.Region
);
     

INSERT INTO FactSales
(
    TransactionID,
    CustomerSK,
    ProductSK,
    StoreSK,
    Quantity,
    Amount,
    TxnDate
)
SELECT
    s.TransactionID,
    dc.CustomerSK,
    dp.ProductSK,
    ds.StoreSK,
    s.Quantity,
    CAST(s.Quantity * dp.UnitPrice AS DECIMAL(10,2)) AS Amount,
    s.TxnDate
FROM training_catalog.silver.silver_sales s
JOIN DimCustomer dc
  ON s.CustomerID = dc.CustomerID
 AND dc.IsActive = 1
JOIN DimProduct dp
  ON s.ProductID = dp.ProductID
JOIN DimStore ds
  ON s.StoreID = ds.StoreID
LEFT JOIN FactSales f
  ON s.TransactionID = f.TransactionID
WHERE f.TransactionID IS NULL;
     

SELECT 'DimCustomer' AS table_name, COUNT(*) FROM DimCustomer
UNION ALL
SELECT 'DimProduct', COUNT(*) FROM DimProduct
UNION ALL
SELECT 'DimStore', COUNT(*) FROM DimStore
UNION ALL
SELECT 'FactSales', COUNT(*) FROM FactSales;